# fynance 2.0 — quickstart

The end-to-end pipeline: **data → features → signal → backtest → metrics → report**.
Every piece is usable standalone; `Strategy` is an optional orchestrator.

In [ ]:
import numpy as np
import fynance as fy

## 1. Data

Load a CSV/Parquet file with `fy.load("prices.csv")`, or build a `PriceSeries` directly.
A `PriceSeries` is a thin, numpy-backed value object (no pandas).

In [ ]:
rng = np.random.default_rng(1)
prices = fy.PriceSeries(100 * np.cumprod(1 + rng.normal(0.0004, 0.01, 750)), name="asset")
prices

## 2. Price → returns, and a quick metric

Metrics live in `fynance.metrics` and operate on a price/equity curve.

In [ ]:
returns = prices.to_returns("log")
print("buy & hold Sharpe:", round(float(fy.sharpe(prices.values)), 3))

## 3. Compose a strategy

A momentum feature → position → vectorized backtest with proportional costs.

In [ ]:
def momentum(p):
    return np.sign(np.diff(p, prepend=p[0]))

strat = fy.Strategy(features=momentum, signal=lambda x: x,
                    cost=fy.ProportionalCost(fee=0.0005))
result = strat.run(prices)
result.summary()

## 4. Report

`tearsheet` is the one-call report (equity, drawdown, rolling Sharpe, metrics table).

In [ ]:
fig = fy.tearsheet(result)
fig

## 5. Walk-forward (no lookahead)

Refit per window on the train slice only, predict out-of-sample, stitch and backtest.

In [ ]:
class LastSign:
    def fit(self, X, y):
        return self

    def predict(self, X):
        X = np.asarray(X)
        return np.sign(np.diff(X, prepend=X[0]))

y = np.sign(np.diff(prices.values, prepend=prices.values[0]))
wf = fy.Strategy(model=LastSign(), signal=fy.sign).run_walk_forward(
    prices, y, train=200, test=50, step=50)
wf.summary()